# 06 - Store Sales Forecasting - Final Candidate Retraining

## Objective

Retrain the selected advanced LightGBM pipeline on the full available training dataset.

This notebook is not used for model comparison or validation.

Its purpose is to:

- reuse the validated advanced feature pipeline
- retrain a single final model on the full training history
- log the final retraining step in MLflow
- prepare the model for Model Registry registration

## Context

Previous notebooks established the following workflow:

- NB4 tracked baseline and advanced experiments with a fixed temporal split
- NB5 introduced walk-forward cross-validation on a recent subset
- NB5 full data became the primary validation reference using recent folds on the full dataset

This notebook materializes the final candidate after validation has already been completed.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from lightgbm import LGBMRegressor

sys.path.append(str(Path().resolve().parent))

from src.features import add_baseline_features, add_advanced_features, encode_family

## Setup

Use the same validated feature pipeline and the same LightGBM configuration selected in the previous notebooks.

Important:
- full dataset is used
- no validation split is performed
- NB5 full-data cross-validation metrics are logged only as reference

In [2]:
SEED = 42
DATA_DIR = Path("../data")

USE_FULL_DATA = True
LAST_N_DAYS = None  # not used in final retraining

MLFLOW_EXPERIMENT_NAME = "store_sales_forecasting"
MLFLOW_TRACKING_URI = "file:../mlruns"
MODEL_NAME = "store_sales_lgbm"

# Reference metrics from NB5 full-data validation
NB5_FULL_RMSLE_MEAN = 0.599880
NB5_FULL_RMSLE_STD = 0.015144
NB5_FULL_MAE_MEAN = 68.447697
NB5_FULL_RMSE_MEAN = 248.706697
NB5_FULL_R2_MEAN = 0.965119

print("Setup OK")
print("Data dir:", DATA_DIR.resolve())
print("Tracking URI:", MLFLOW_TRACKING_URI)
print("Use full data:", USE_FULL_DATA)
print("Registered model name:", MODEL_NAME)

Setup OK
Data dir: /home/donatocorbacio/projects/store-sales-project/data
Tracking URI: file:../mlruns
Use full data: True
Registered model name: store_sales_lgbm


## MLflow setup

Configure the experiment used to track the final retraining step.

In [3]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("Active experiment:", MLFLOW_EXPERIMENT_NAME)

Active experiment: store_sales_forecasting


## Load data

Load the Store Sales training and test datasets and inspect their temporal coverage.

In [4]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train range:", train["date"].min(), "->", train["date"].max())
print("Test range:", test["date"].min(), "->", test["date"].max())

Train shape: (3000888, 6)
Test shape: (28512, 5)
Train range: 2013-01-01 00:00:00 -> 2017-08-15 00:00:00
Test range: 2017-08-16 00:00:00 -> 2017-08-31 00:00:00


## Sorting

Sort each `(store_nbr, family)` series by date before generating lag and rolling features.

This preserves temporal consistency.

In [5]:
train = train.sort_values(["store_nbr", "family", "date"]).copy()
test = test.sort_values(["store_nbr", "family", "date"]).copy()

print("Sorting completed.")
display(train.head())

Sorting completed.


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0


## Dataset scope

Use the full training history.

The subset logic from NB1–NB5 is disabled here.

In [6]:
if not USE_FULL_DATA:
    cutoff = train["date"].max() - pd.Timedelta(days=LAST_N_DAYS)
    train = train[train["date"] >= cutoff].copy()

print("Train shape after scope selection:", train.shape)
print("Date range after scope selection:", train["date"].min(), "->", train["date"].max())

Train shape after scope selection: (3000888, 6)
Date range after scope selection: 2013-01-01 00:00:00 -> 2017-08-15 00:00:00


## Calendar features

Create the same calendar features used in the previous notebooks.

In [7]:
for df in [train, test]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

display(train[["date", "year", "month", "day", "dayofweek", "weekofyear", "is_weekend"]].head())

,date,year,month,day,dayofweek,weekofyear,is_weekend
0,2013-01-01,2013,1,1,1,1,0
1782,2013-01-02,2013,1,2,2,1,0
3564,2013-01-03,2013,1,3,3,1,0
5346,2013-01-04,2013,1,4,4,1,0
7128,2013-01-05,2013,1,5,5,1,1


## Rebuild advanced feature pipeline

Reuse the exact advanced feature engineering pipeline introduced in NB3 and validated in NB4 / NB5 / NB5 full data.

No feature changes are introduced in this notebook.

In [8]:
advanced_train = add_baseline_features(train)
advanced_train = add_advanced_features(advanced_train)
advanced_train = advanced_train.dropna().copy()

advanced_train, advanced_test, family_mapping = encode_family(advanced_train, test)

advanced_features = [
    "store_nbr",
    "family",
    "onpromotion",
    "year",
    "month",
    "day",
    "dayofweek",
    "weekofyear",
    "is_weekend",
    "lag_1",
    "lag_7",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_14",
    "trend_1_7",
    "promo_last_7",
]

print("Advanced dataset shape:", advanced_train.shape)
print("Number of features:", len(advanced_features))
print("Advanced date range:", advanced_train["date"].min(), "->", advanced_train["date"].max())

Advanced dataset shape: (2975940, 19)
Number of features: 16
Advanced date range: 2013-01-15 00:00:00 -> 2017-08-15 00:00:00


## Final training dataset

At this stage, model selection and validation are already completed.

Therefore:
- no temporal split is used
- no cross-validation is performed
- the final candidate is trained on the full advanced dataset

In [9]:
X_train_final = advanced_train[advanced_features]
y_train_final = advanced_train["sales"]

print("Final training shape:", X_train_final.shape)
print("Target shape:", y_train_final.shape)

Final training shape: (2975940, 16)
Target shape: (2975940,)


## Final model configuration

Keep the same LightGBM configuration validated previously.

In [10]:
final_params = {
    "model_type": "LGBMRegressor",
    "feature_set": "advanced_v1",
    "training_stage": "final_retraining",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "random_state": SEED,
    "use_full_data": USE_FULL_DATA,
    "n_features": len(advanced_features),
}

final_params

{'model_type': 'LGBMRegressor',
 'feature_set': 'advanced_v1',
 'training_stage': 'final_retraining',
 'n_estimators': 300,
 'learning_rate': 0.05,
 'num_leaves': 31,
 'random_state': 42,
 'use_full_data': True,
 'n_features': 16}

## Final retraining

Train the final candidate model on the full dataset.

Important:
- no new validation metrics are computed here
- NB5 full-data metrics are logged only as reference evidence

In [11]:
with mlflow.start_run(run_name="final_lgbm_advanced_v1_full_retraining"):
    mlflow.set_tags({
        "project": "store_sales_forecasting",
        "notebook": "nb6",
        "model_family": "lightgbm",
        "experiment_type": "final_retraining",
        "training_scope": "full_data",
        "validation_reference": "nb5_full_data_walk_forward_cv",
        "primary_metric": "reference_nb5_full_rmsle_mean",
        "registry_candidate": "true",
    })

    mlflow.log_params(final_params)

    # reference metrics from NB5 full-data validation
    mlflow.log_metric("reference_nb5_full_rmsle_mean", NB5_FULL_RMSLE_MEAN)
    mlflow.log_metric("reference_nb5_full_rmsle_std", NB5_FULL_RMSLE_STD)
    mlflow.log_metric("reference_nb5_full_mae_mean", NB5_FULL_MAE_MEAN)
    mlflow.log_metric("reference_nb5_full_rmse_mean", NB5_FULL_RMSE_MEAN)
    mlflow.log_metric("reference_nb5_full_r2_mean", NB5_FULL_R2_MEAN)

    final_model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=SEED,
        n_jobs=-1
    )

    final_model.fit(X_train_final, y_train_final)

    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="final_model"
    )

    run_id = mlflow.active_run().info.run_id
    model_uri = f"runs:/{run_id}/final_model"

    print("Final candidate retrained successfully.")
    print("Run ID:", run_id)
    print("Model URI:", model_uri)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.221089 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2190
[LightGBM] [Info] Number of data points in the train set: 2975940, number of used features: 16
[LightGBM] [Info] Start training from score 359.135740


2026/04/23 15:14:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 15:14:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final candidate retrained successfully.
Run ID: acf13cf61bdd46deb0b756f94f1e26fc
Model URI: runs:/acf13cf61bdd46deb0b756f94f1e26fc/final_model


## Conclusion

This notebook finalizes the forecasting workflow by retraining the selected advanced LightGBM pipeline on the full available training history.

Key points:

- the advanced feature pipeline is reused without modification
- the model is retrained on the full dataset
- no new validation metrics are computed at this stage
- NB5 full-data cross-validation remains the primary validation reference
- the retrained candidate is registered in the Model Registry as the next model version

This notebook represents the production-oriented finalization step of the project.